In [1]:
!pip install --upgrade -q transformers
!pip install -q bitsandbytes accelerate qwen_vl_utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 63.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 94.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 53.0 MB/s eta 0:00:00:00:0100:01


In [2]:
import pandas as pd
import os

In [3]:
BASE_FOLDER = "path/to/images"
IMGS = os.path.join(BASE_FOLDER, "omnigen2")
CAPTION_CSV = os.path.join(BASE_FOLDER, "evaluation_data.csv")

In [4]:
df = pd.read_csv(CAPTION_CSV)

In [5]:
import torch
from PIL import Image
from tqdm import tqdm
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
from typing import List, Union

# ---------------------------------------------------------
# 1. Load Model & Processor (High-Speed Config)
# ---------------------------------------------------------
model_id = "Qwen/Qwen3-VL-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16 # float16 for Kaggle T4
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" 
)

# Cap image resolution to drastically speed up processing
processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28 
)

# Required for batched generation
processor.tokenizer.padding_side = "left"
model.generation_config.pad_token_id = processor.tokenizer.eos_token_id

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [6]:
# ---------------------------------------------------------
# 2. Define the Batched Evaluation Function
# ---------------------------------------------------------
def critique_images_batched(
    image_paths: Union[str, List[str]], 
    target_prompts: Union[str, List[str]], 
    model: Qwen3VLForConditionalGeneration, 
    processor: AutoProcessor,
    batch_size: int = 4
) -> List[str]:
    
    if isinstance(image_paths, str):
        image_paths = [image_paths]
    
    if isinstance(target_prompts, str):
        target_prompts = [target_prompts] * len(image_paths)
        
    if len(image_paths) != len(target_prompts):
        raise ValueError("Paths and prompts must match in length.")

    all_results = []

    # Process in chunks using batch_size
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Critiquing Batches"):
        batch_paths = image_paths[i : i + batch_size]
        batch_prompts = target_prompts[i : i + batch_size]
        
        batch_messages = []
        
        # Build the message payload for the entire batch
        for img_path, prompt in zip(batch_paths, batch_prompts):
            critic_prompt = f"""
            You are a visual-language critic. An image was generated by a diffusion model using this prompt:
            {prompt}
            Analyze the image relative to the prompt.
            Please:
            1. Identify any deviations from the prompt and explain how they differ.
            2. Note which keywords or phrases are poorly represented or misinterpreted.

            Provide a clear, structured critique in a paragraph.
            """

            msg = [{
                "role": "user",
                "content": [
                    {"type": "image", "image": img_path},
                    {"type": "text", "text": critic_prompt},
                ],
            }]
            batch_messages.append(msg)

        # Apply chat templates
        texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch_messages]
        
        # Use Qwen's specific vision info processor
        image_inputs, video_inputs = process_vision_info(batch_messages)
        
        # Tokenize and pad the batch
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        # Generate critiques
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=1024)

        # Trim prompts and decode
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        batch_outputs = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        
        all_results.extend([out.strip() for out in batch_outputs])
        
        # Prevent VRAM creeping
        del inputs, generated_ids, generated_ids_trimmed
        torch.cuda.empty_cache()

    return all_results

In [7]:
my_images = [
    os.path.join(IMGS, str(img) + ".png")
    for img in df["image_name"]
]

my_prompts = list(df["description"])

In [8]:
# Run the batch
critiques = critique_images_batched(my_images, my_prompts, model, processor)

Critiquing Batches: 100%|██████████| 13/13 [1:09:12<00:00, 319.44s/it]


In [9]:
# Print results
for i, critique in enumerate(critiques):
    print(f"\n--- Critique for Image {i+1} ---")
    print(critique)


--- Critique for Image 1 ---
The image deviates significantly from the prompt’s botanical accuracy and fails to represent key structural and textual elements expected in a scientific illustration. While the prompt describes a racemose inflorescence with apical growth, acropetal flower development, and compound pinnate leaves, the image instead shows a single-stemmed, upright floral arrangement with flowers appearing in a relatively uniform, dense cluster along the stem—more consistent with a corymbose or panicle-like inflorescence than a true raceme, which typically has flowers spaced along an elongating axis with pedicels of varying lengths. The leaves depicted are compound pinnate (as described), but they appear disproportionately large and stylized, with a near-symmetrical, almost star-shaped arrangement that lacks the natural asymmetry and venation detail expected in botanical illustrations. The text below the image is gibberish (“The clmecalts any th lusk cull oil...”), indicatin

In [11]:
import csv


with open("critics.csv", "w") as f:
    writer = csv.writer(f)
    for critic, img in zip(critiques, list(df["image_name"])):
        writer.writerow([str(img)+".png", critic])